In [2]:
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import inspect
from tqdm import tqdm

In [1]:
%pip install tqdm

     ---------------------------------------- 77.1/77.1 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3.1 -> 23.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
batch_size = 32

In [8]:
model_dictionary = {m[0]:m[1] for m in inspect.getmembers(tf.keras.applications, inspect.isfunction)}

In [15]:
train = tf.keras.utils.image_dataset_from_directory(
  "Bees/train",
  validation_split=0.2,
  subset="training",
  seed=123,
  batch_size=batch_size)

validation = tf.keras.utils.image_dataset_from_directory(
  "Bees/test",
  validation_split=0.2,
  subset="validation",
  seed=123,
  batch_size=batch_size)

class_names = train.class_names

Found 0 files belonging to 0 classes.
Using 0 files for training.


ValueError: No images found in directory Bees/train. Allowed formats: ('.bmp', '.gif', '.jpeg', '.jpg', '.png')

In [ ]:
num_train = 800
num_validation = 199
num_classes = 2
num_iterations = int(num_train/batch_size)

print(f'Num train images: {num_train} \
        \nNum validation images: {num_validation} \
        \nNum classes: {num_classes} \
        \nNum iterations per epoch: {num_iterations}')

In [ ]:
def normalize_img(image, label, img_size):
    # Resize image to the desired img_size and normalize it
    # One hot encode the label
    image = tf.image.resize(image, img_size)
    image = tf.cast(image, tf.float32)/255.
    label = tf.one_hot(label, depth=num_classes)
    return image, label

def preprocess_data(train, validation, img_size):
    # Apply the normalize_img function on all train and validation data and create batches
    train_processed = train.map(lambda image, label: normalize_img(image, label, img_size))

    # If your data is already batched (eg, when using the image_dataset_from_directory function), remove .batch(batch_size)
    train_processed = train_processed.repeat()

    validation_processed = validation.map(lambda image, label: normalize_img(image, label, img_size))

    # If your data is already batched (eg, when using the image_dataset_from_directory function), remove .batch(batch_size)
    #validation_processed = validation_processed

    return train_processed, validation_processed

In [ ]:
train_processed_224, validation_processed_224 = preprocess_data(train, validation, img_size=[224,224])
train_processed_331, validation_processed_331 = preprocess_data(train, validation, img_size=[331,331])

In [ ]:
model_benchmarks = {'model_name': [], 'num_model_params': [], 'validation_accuracy': []}

for model_name, model in tqdm(model_dictionary.items()):
    # Special hadling for "NASNetLarge" since it require input images with size (331,331)
    if 'NASNetLarge' in model_name:
        input_shape=(331, 331, 3)
        train_processed = train_processed_331
        validation_processed = validation_processed_331
    else:
        input_shape=(224,224,3)
        train_processed = train_processed_224
        validation_processed = validation_processed_224

    # Load the pre-trained model with global average pooling as the last layerand freeze the modelweights
    pre_trained_model = model(include_top=False, pooling='avg', input_shape=input_shape)
    pre_trained_model.trainable = False

    # Custom modifications on top of pre-trained model and fit
    clf_model = tf.keras.models.Sequential()
    clf_model.add(pre_trained_model)
    clf_model.add(tf.keras.layers.Dense(num_classes, activation='softmax'))
    clf_model.compile(loss='categorical_crossentropy', metrics=['accuracy'])
    history = clf_model.fit(train_processed, epochs=3, validation_data=validation_processed, steps_per_epoch=num_iterations)

    # Calculate all relevant metrics
    model_benchmarks['model_name'].append(model_name)
    model_benchmarks['num_model_params'].append(pre_trained_model.count_params())
    model_benchmarks['validation_accuracy'].append(history.history['val_accuracy'][-1])

In [ ]:
# Convert Results to DataFrame for easy viewing
benchmark_df = pd.DataFrame(model_benchmarks)

# Sort in ascending order of num_model_params column
benchmark_df.sort_values('validation_accuracy', inplace=True)

# Write results to csv file
benchmark_df.to_csv('couvre_chef_benchmark.csv', index=False)
benchmark_df

In [ ]:
benchmark_df = pd.read_csv('couvre_chef_benchmark.csv', sep=',')
benchmark_df = benchmark_df.sort_values(by='validation_accuracy', ascending=False)
benchmark_df = benchmark_df[:10]

plt.figurefigsize=(7,5)
for row in benchmark_df.itertuples():
    plt.scatter(row.num_model_params, row.validation_accuracy, label=row.model_name, s=150, linewidths=2)
    
plt.xscale('log')
plt.xlabel('Number of Parameters in Model')
plt.ylabel('Validation Accuracy after 3 Epochs')
plt.title('Accuracy vs Model Size')

plt.legend(bbox_to_anchor=(1,1), loc='upper left')

In [ ]:
benchmark_df = pd.read_csv('couvre_chef_benchmark.csv', sep=',')
benchmark_df = benchmark_df.sort_values(by='num_model_params', ascending=True)
benchmark_df = benchmark_df[:10]

plt.figurefigsize=(7,5)
for row in benchmark_df.itertuples():
    plt.scatter(row.num_model_params, row.validation_accuracy, label=row.model_name, s=150, linewidths=2)
    
plt.xscale('log')
plt.xlabel('Number of Parameters in Model')
plt.ylabel('Validation Accuracy after 3 Epochs')
plt.title('Accuracy vs Model Size')

plt.legend(bbox_to_anchor=(1,1), loc='upper left')